## Mutual Fund Analysis -

In [5]:
import os
import requests
import pandas as pd
import json
from datetime import datetime as dt, timedelta as td


amfi_base_url = "https://www.amfiindia.com/"
amfi_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Origin": "https://www.amfiindia.com",
    "Referer": "https://www.amfiindia.com/"
}

def amfi_api_call(full_url: str, payload: dict, method: str, headers: dict = amfi_headers):
    """Generic function to make API calls to AMFI endpoints.
    Args:        
        full_url (str): The complete URL for the API endpoint.
        payload (dict): The JSON payload to be sent in the POST request.
        method (str): The HTTP method to use for the request.
        headers (dict): The headers to be included in the request. Defaults to amfi_headers.
    Returns:        
        dict: The JSON response from the API if the call is successful, None otherwise.
    """
    if method == "POST":
        response = requests.post(url=full_url,
                                headers=headers,
                                json=payload)
    else:
        response = requests.get(url=full_url,
                                headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"API call failed with status code: {response.status_code}")
        return None

### Mutual Fund Performance Data -

In [6]:
# fund_performance_web_url = "https://www.amfiindia.com/otherdata/fund-performance"

is_holiday_api_url = "https://www.amfiindia.com/gateway/pollingsebi/api/amfi/isHoliday"
fund_performance_filter_api_url = "https://www.amfiindia.com/gateway/pollingsebi/api/amfi/fundperformancefilters"
fund_performance_subcategory_api_url = "https://www.amfiindia.com/gateway/pollingsebi/api/amfi/getsubcategory"
fund_performance_api_url = "https://www.amfiindia.com/gateway/pollingsebi/api/amfi/fundperformance"

fund_performance_filter_out = amfi_api_call(full_url=fund_performance_filter_api_url, payload={}, method="POST")
final_output = fund_performance_filter_out.get("data")
fund_performance_filter_dfs = []
fund_performance_subcategory_dfs = []
for key, value in final_output.items():
    if key in ["maturityTypeList", "investmentTypeList", "mutualFundList"]:
        df = pd.DataFrame(value)
        df["filter_category"] = key
        fund_performance_filter_dfs.append(df)
    if key == "investmentTypeList":
        for item in value:
            fund_performance_subcategory_out = amfi_api_call(full_url=fund_performance_subcategory_api_url, payload={"category": item.get('id')}, method="POST")
            fund_performance_subcategory_df = pd.DataFrame(fund_performance_subcategory_out.get("data"))
            fund_performance_subcategory_df["investment_type_id"] = item.get('id')
            fund_performance_subcategory_df["investment_type_name"] = item.get('name')
            fund_performance_subcategory_dfs.append(fund_performance_subcategory_df)
report_date = final_output.get("reportDate")
final_fund_performance_filter_df = pd.concat(fund_performance_filter_dfs, ignore_index=True)
final_fund_performance_subcategory_dfs = pd.concat(fund_performance_subcategory_dfs, ignore_index=True)
# fund_performance_out = amfi_api_call(full_url=fund_performance_api_url, payload={"maturityType": 1, "category": 1, "subCategory": 1, "mfid": 0, "reportDate": report_date}, method="POST")
fund_performance_dfs = []
payload = {"maturityType": 1, "category": 1, "subCategory": 1, "mfid": 0, "reportDate": report_date}
for maturity_type in final_fund_performance_filter_df[final_fund_performance_filter_df['filter_category'] == 'maturityTypeList'].itertuples(index=False):
    # print(maturity_type.id, maturity_type.name, maturity_type.filter_category)
    for category in final_fund_performance_filter_df[final_fund_performance_filter_df['filter_category'] == 'investmentTypeList'].itertuples(index=False):
        # print(category.id, category.name, category.filter_category)
        for sub_category in final_fund_performance_subcategory_dfs[final_fund_performance_subcategory_dfs['investment_type_id'] == category.id].itertuples(index=False):
            # print(sub_category.id, sub_category.name, sub_category.investment_type_id, sub_category.investment_type_name)
            payload["maturityType"] = maturity_type.id
            payload["category"] = category.id
            payload["subCategory"] = sub_category.id
            fund_performance_out = amfi_api_call(full_url=fund_performance_api_url, payload=payload, method="POST")
            fund_performance_df = pd.DataFrame(fund_performance_out.get("data"))
            fund_performance_df['maturity_type'] = maturity_type.name
            fund_performance_df['investment_type'] = category.name
            fund_performance_df['sub_category'] = sub_category.name
            fund_performance_dfs.append(fund_performance_df)
            # print(f"Payload: {payload}")
final_fund_performance_df = pd.concat(fund_performance_dfs, ignore_index=True)

In [7]:
print(f"Report Date: {report_date}")
# print(final_fund_performance_filter_df.shape)
# print(final_fund_performance_subcategory_dfs.shape)
print(f"Final Fund Performance DataFrame Size: {final_fund_performance_df.shape}")

Report Date: 03-Jun-2026
Final Fund Performance DataFrame Size: (2045, 56)


In [8]:
def get_amfi_mutual_fund_id():
    # https://www.amfiindia.com/api/amc-adresses?page=1&pageSize=12&MF_ID=53
    # https://www.amfiindia.com/api/amc-adresses?page=1&pageSize=12
    # https://www.amfiindia.com/api/amc-adresses?page=1&pageSize=12&MF_ID=53&City=ETAWAH
    full_url = amfi_base_url + f"api/amc-adresses?page=1&pageSize=1"
    try:
        response = requests.get(full_url, headers=amfi_headers)
        data = response.json().get("amcs")
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return False

In [9]:
out = get_amfi_mutual_fund_id()
print(out)

    mf_id                                            mf_name
0      62                   360 ONE Asset Management Limited
1      85        Abakkus Investment Managers Private Limited
2       3                  Aditya Birla Sun Life AMC Limited
3      86    AlphaGrep Investment Management Private Limited
4      80         Angel One Asset Management Company Limited
5      87               ASK ASSET MANAGEMENT PRIVATE LIMITED
6      53                     Axis Asset Management Co. Ltd.
7      75             Bajaj Finserv Asset Management Limited
8      48                                Bandhan AMC Limited
9      46  Bank of India Investment Managers Private Limited
10      4  Baroda BNP Paribas Asset Management India Priv...
11     32     Canara Robeco Asset Management Company Limited
12     81       Capitalmind Asset Management Private Limited
13     84                         Choice AMC Private Limited
14      6                 DSP Asset Managers Private Limited
15     47               

In [15]:
fund_performance_out

{'validationStatus': 'SUCCESS',
 'validationMsg': 'SUCCESS',
 'errorMsgs': [],
 'data': []}

In [14]:
final_fund_performance_df.columns

Index(['preNavDate', 'preNavRegular', 'preNavDirect', 'specialCharAum',
       'schemeName', 'benchmark', 'riskometerScheme', 'riskometerBenchmark',
       'navDate', 'navRegular', 'navDirect', 'return7DaysRegular',
       'return7DaysDirect', 'return7DaysBenchmark', 'return15DaysRegular',
       'return15DaysDirect', 'return15DaysBenchmark', 'return1MonthRegular',
       'return1MonthDirect', 'return1MonthBenchmark', 'return3MonthRegular',
       'return3MonthDirect', 'return3MonthBenchmark', 'return6MonthRegular',
       'return6MonthDirect', 'return6MonthBenchmark', 'return1YearRegular',
       'return1YearDirect', 'return1YearBenchmark', 'return3YearRegular',
       'return3YearDirect', 'return3YearBenchmark', 'return5YearRegular',
       'return5YearDirect', 'return5YearBenchmark', 'return10YearRegular',
       'return10YearDirect', 'return10YearBenchmark',
       'returnSinceLaunchRegular', 'returnSinceLaunchDirect',
       'returnSinceLaunchBenchmarkRegular', 'returnSinceLaunchB

#### Returns Analysis -

In [12]:
# Returns Analysis
final_fund_performance_df.loc[:, ['schemeName', 'benchmark', 'navDate', 'navRegular', 'navDirect', 'return1YearDirect', 
                                  'return3YearDirect', 'return5YearDirect', 'return10YearDirect']]

,schemeName,benchmark,navDate,navRegular,navDirect,return1YearDirect,return3YearDirect,return5YearDirect,return10YearDirect
0,Aditya Birla Sun Life Large Cap Fund,Nifty 100 TRI,03-Jun-2026,495.4200,550.6600,-3.357377,11.598158,11.203949,12.37564
1,Axis Large Cap Fund,BSE 100 TRI,03-Jun-2026,56.4600,65.4400,-3.764706,9.418102,7.44998,12.482704
2,Bajaj Finserv Large Cap Fund,Nifty 100 TRI,03-Jun-2026,9.6430,9.9040,0.395337,NaN,NaN,NaN
3,Bandhan Large Cap Fund,BSE 100 TRI,03-Jun-2026,74.2610,86.1600,0.479306,13.994,12.344099,13.456414
4,Bank of India Large Cap Fund,Nifty 100 TRI,03-Jun-2026,16.0800,17.1400,5.217925,15.174499,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2040,SBI Fixed Maturity Plan (FMP) - Series 61,CRISIL Medium to Long Duration Debt A-III Index,03-Jun-2026,13.0353,13.0999,5.294505,7.243204,NaN,NaN
2041,SBI Fixed Maturity Plan (FMP) - Series 67,CRISIL Medium to Long Duration Debt A-III Index,03-Jun-2026,12.9871,13.0425,5.684304,7.024086,NaN,NaN
2042,TRUSTMF Fixed Maturity Plan - Series II (1196 ...,CRISIL Medium Duration Debt A-III Index,03-Jun-2026,1287.3332,1291.6306,7.111679,8.383575,NaN,NaN
2043,UTI Annual Interval Fund - I,NIFTY Low Duration Debt Index A-I,03-Jun-2026,36.0256,36.4851,5.329238,6.31386,6.349509,5.665875


#### Fund Performance Analysis -

In [13]:
# Fund Performance Analysis
final_fund_performance_df.loc[:, ['schemeName', 'navDate', 'dailyAUM', 'ir1YrDirect', 'ir3YrDirect', 'ir5YrDirect', 'ir10YrDirect']]

,schemeName,navDate,dailyAUM,ir1YrDirect,ir3YrDirect,ir5YrDirect,ir10YrDirect
0,Aditya Birla Sun Life Large Cap Fund,03-Jun-2026,28364.697000,-0.855509,0.277464,0.422262,-0.151769
1,Axis Large Cap Fund,03-Jun-2026,29785.535000,-0.674462,-0.471734,-0.925877,-0.160001
2,Bajaj Finserv Large Cap Fund,03-Jun-2026,1460.130000,0.840485,NaN,NaN,NaN
3,Bandhan Large Cap Fund,03-Jun-2026,1989.305400,1.456496,0.931002,0.506737,0.094227
4,Bank of India Large Cap Fund,03-Jun-2026,211.320000,2.187006,0.910216,NaN,NaN
...,...,...,...,...,...,...,...
2040,SBI Fixed Maturity Plan (FMP) - Series 61,03-Jun-2026,358.852100,None,None,None,None
2041,SBI Fixed Maturity Plan (FMP) - Series 67,03-Jun-2026,624.532800,None,None,None,None
2042,TRUSTMF Fixed Maturity Plan - Series II (1196 ...,03-Jun-2026,65.280000,None,None,None,None
2043,UTI Annual Interval Fund - I,03-Jun-2026,21.974974,None,None,None,None


In [ ]:
# https://www.amfiindia.com/investor/become-mf-distributor?zoneName=trackYourMF
# https://www.amfiindia.com/investor/knowledge-center-info?zoneName=expenseRatio
# https://www.amfiindia.com/otherdata/fund-performance/information-ratio